# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring the **FAIR<sup>2</sup>** protein mass spectrometry dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source
The dataset source is provided via a [Croissant schema](https://mlcommons.org/croissant/) URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading

Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# Access and print metadata summary
metadata = dataset.metadata

print(f"Dataset name: {metadata.name}\n")
print(f"Description: {metadata.description}\n")
print(f"Published: {metadata.datePublished}")
if hasattr(metadata, 'keywords'):
    print(f"Keywords: {', '.join(metadata.keywords)}")


## 2. Data Overview

Explore available record sets, and list their fields and IDs. All Croissant entities (record sets, fields, etc.) are referred to by their `@id`.

In [ ]:
# List the record sets in the dataset
print(f"Record sets in the dataset:")
for rs in metadata.recordSet:
    print(f"- @id: {rs['@id']} | name: {rs.get('name', '')}")

# For illustration, let's choose the main protein records set.
# In this schema, the following record set @id contains the quantification results:
main_record_set_id = 'https://api.app.sen.science/frontiers/7154140/e9612b00-2c94-4b51-a4c3-9815525a1694'

# List fields (columns) of this main record set
main_rs = next(rs for rs in metadata.recordSet if rs['@id'] == main_record_set_id)
print(f"\nFields in record set {main_record_set_id}:")
for field in main_rs['field']:
    print(f"- @id: {field['@id']}, name: {field.get('name', '')}, dataType: {field.get('dataType', '')}")

## 3. Data Extraction

Load data from a specific record set into a DataFrame for analysis. Use record set and field `@id`s from the above overview.

In [ ]:
# Define the record set IDs to load
record_set_ids = [
    'https://api.app.sen.science/frontiers/7154140/e9612b00-2c94-4b51-a4c3-9815525a1694',  # Main protein quantification table
    # Add more record set @ids here if desired
]

dataframes = {}
for record_set_id in record_set_ids:
    print(f"Loading records for record set {record_set_id}")
    recs = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(recs)
    dataframes[record_set_id] = df
    print(f"- Columns in this DataFrame: {df.columns.tolist()}\n")

# Show the first 5 records of the main protein table
main_df = dataframes[record_set_ids[0]]
main_df.head()

## 4. Exploratory Data Analysis (EDA)

Let's filter and analyze the data. For example, we'll filter proteins with coverage > 10%, normalize the coverage, and group by protein class if available. 

> Note:
>
> Please refer to the field `@id`s. Here we use:
> - Coverage field: `https://api.app.sen.science/frontiers/7154140/2c1d1147-cfc2-4b99-b1ca-a504e5d456cf` (name: `Coverage (%)`)
> - Protein class field: `https://api.app.sen.science/frontiers/7154140/dffee6f2-6e25-4c28-baa7-376f5f54e81e` (if present)

In [ ]:
main_record_set_id = record_set_ids[0]  # The main protein quantification table

# Field @id for coverage—a typical candidate for filtering
coverage_field = 'https://api.app.sen.science/frontiers/7154140/2c1d1147-cfc2-4b99-b1ca-a504e5d456cf'

# Field @id for protein class, if present
protein_class_field = 'https://api.app.sen.science/frontiers/7154140/dffee6f2-6e25-4c28-baa7-376f5f54e81e'

df = dataframes[main_record_set_id]

# Clean and convert coverage to float if it's not already
df[coverage_field] = pd.to_numeric(df[coverage_field], errors='coerce')

# Filter proteins with coverage > 10
threshold = 10.0
filtered_df = df[df[coverage_field] > threshold].copy()
print(f"Records with coverage > {threshold} (field @id: {coverage_field}): {len(filtered_df)}\n")
print(filtered_df[[coverage_field]].head())

# Normalize coverage
filtered_df[f"{coverage_field}_normalized"] = (filtered_df[coverage_field] - filtered_df[coverage_field].mean()) / filtered_df[coverage_field].std()
print(f"\nNormalized coverage (")
print(filtered_df[[coverage_field, f"{coverage_field}_normalized"]].head())

# If protein class field exists, group by it
if protein_class_field in df.columns:
    grouped_df = filtered_df.groupby(protein_class_field).mean(numeric_only=True)
    print(f"\nGrouped by protein class (field @id: {protein_class_field}):")
    print(grouped_df[[coverage_field, f"{coverage_field}_normalized"]].head())

## 5. Visualization

Let's plot the distribution of the protein coverage percentage.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style='whitegrid')

plt.figure(figsize=(8, 4))
sns.histplot(df[coverage_field].dropna(), bins=30, kde=True)
plt.title('Distribution of Protein Coverage (%)')
plt.xlabel('Coverage (%)')
plt.ylabel('Count')
plt.show()


## 6. Conclusion

- This notebook demonstrated how to explore a mass spectrometry protein dataset using the `mlcroissant` library.
- We loaded dataset metadata, inspected record sets and fields by their Croissant `@id`, and performed basic filtering, normalization, and visualization steps for one quantitative field (`Coverage (%)`).
- This workflow can be adapted to explore other fields or record sets using their respective IDs for reproducible FAIR data analysis.